# as-strided-windowing — ex1: compute size + stride args for a 1-D sliding window

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-windowing`. Running the final beacon cell reports progress against the `PyTorch: as_strided windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: as_strided windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-windowing`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-windowing"
DD_SUBTOPIC = "PyTorch: as_strided windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `as_strided` windowing — quick refresher

`t.as_strided(x, size, stride)` builds a **zero-copy** view at exactly the `(size, stride)` you specify. The classic sliding-window trick is to set both the window-position stride *and* the within-window stride to the source's element stride, so the view becomes a (n_windows, window_size) matrix that aliases the source.

**Pattern for 1-D sliding window of width `K` over a 1-D tensor `x` of length `L`:**
```python
sL, = x.stride()
windows = t.as_strided(x, size=(L - K + 1, K), stride=(sL, sL))
```

**Generalises to N-D inputs.** For batched/channelled input `(B, IC, W)`, you pull all of `x.stride()` and build a `(B, IC, L_out, K)` view with stride `(sB, sIC, sW, sW)`. This is exactly the ARENA `conv1d_minimal` trick.

**Why you need the source's stride, not `1`.** If `x` was itself created via `permute`/`transpose`/another `as_strided` call, its last stride may be larger than 1. Hard-coding `1` will silently scan the wrong memory cells.

### Exercise 1 — compute size + stride args for a 1-D sliding window

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Compute the `size` and `stride` arguments needed to pass to `t.as_strided` for a width-`K` sliding window over a 1-D tensor, without doing the windowing itself.
> Keywords: sliding-window, stride-math, as_strided
> ```

**KCs targeted:** `as-strided-window-size`, `as-strided-window-stride`

Implement `ex1_window_args(x, kernel_width)`. Given a 1-D tensor `x` and an integer `kernel_width`, return the `(size, stride)` tuple of tuples you'd pass to `t.as_strided` to produce the sliding-window view used in ARENA's `conv1d_minimal`.

Specifically, for `x` of length `L`:
- `size` = `(L - kernel_width + 1, kernel_width)`
- `stride` = `(x.stride(0), x.stride(0))` — **both axes use the source's stride**, not `1`. (If `x` was already non-contiguous, hard-coding `1` reads the wrong memory.)

Return: `(size_tuple, stride_tuple)` — both tuples of `int`.

**Don't actually call `as_strided` here** — just compute the args. The test will pass them into `as_strided` and verify the resulting window matches what convolution prep expects.

In [ ]:
def ex1_window_args(x: Tensor, kernel_width: int) -> tuple:
    L = x.shape[0]
    s, = x.stride()
    size = (L - kernel_width + 1, kernel_width)
    stride = (s, s)
    return size, stride


<details><summary>Solution</summary>

```python
def ex1_window_args(x: Tensor, kernel_width: int) -> tuple:
    L = x.shape[0]
    s, = x.stride()
    size = (L - kernel_width + 1, kernel_width)
    stride = (s, s)
    return size, stride
```

**Both strides are the source's element stride.** The first stride says 'how to advance one window position' — one element forward through `x`. The second stride says 'how to advance one step inside a window' — also one element forward through `x`. Same value.

**Why we don't hard-code `1`.** If `x` was built via `x = source[::2]`, its element stride is `2`, not `1`. Hard-coding `1` would scan adjacent memory cells — silently giving you the WRONG windowed view with no error message.

**Generalises directly to N-D.** For ARENA's 1-D conv with batch + channels, you pull all of `x.stride()` (three values) and produce a 4-tuple stride `(s_B, s_IC, s_W, s_W)`. Same logic — window position and within-window both advance one spatial step.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()